# Agent 1 - Module 2: Self-Contained Semantic Chunking

## Final production workflow

```text
OUTPUT/<transcript>/01_preprocessing.pdf
        -> extract final cleaned transcript only
        -> sentence splitting
        -> semantic-unit construction
        -> MiniLM embeddings
        -> neighbour similarity
        -> adaptive threshold
        -> guarded semantic boundaries
        -> max-size splitting and conditional overlap
        -> logical segment and continuation metadata
        -> OUTPUT/<transcript>/02_chunking.pdf
        -> OUTPUT/<transcript>/02_chunking.json
```

This notebook contains the full Module 2 implementation. It does **not** import Module 2 logic from project `.py` files. Therefore, Module 2 no longer needs:

- `app/schemas/chunk.py`
- `app/services/semantic_chunker.py`
- Module 2 test scripts migrated into this notebook

The notebook also embeds the required embedding service. However, the current `app/services/embedding_service.py` is shared with Module 3, so keep that file until Module 3 has also been migrated to its own self-contained notebook.

The final batch uses `sentence-transformers/all-MiniLM-L6-v2`. The historical Qwen comparison remains available as an optional experiment and is disabled by default.

# 0. Environment and project paths

Keep this notebook inside `Agent_1/Notebooks` and run **Restart Kernel -> Run All**.

In [1]:
from __future__ import annotations

import importlib
import json
import os
import re
import subprocess
import sys
import time
from collections import Counter
from datetime import datetime, timezone
from html import escape
from pathlib import Path
from typing import Any


def find_project_root(start: Path) -> Path:
    candidates: list[Path] = []
    for candidate in [
        start,
        *start.parents,
        start / "Agent_1",
        *[parent / "Agent_1" for parent in start.parents],
    ]:
        candidate = candidate.resolve()
        if candidate not in candidates:
            candidates.append(candidate)

    for candidate in candidates:
        if (
            (candidate / "OUTPUT").is_dir()
            and (
                (candidate / "Notebooks").is_dir()
                or (candidate / "requirements.txt").is_file()
            )
        ):
            return candidate

    raise RuntimeError(
        "Agent_1 project root was not found. Keep this notebook inside "
        "Agent_1/Notebooks and ensure Agent_1/OUTPUT exists."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
OUTPUT_DIR = PROJECT_ROOT / "OUTPUT"

# Install only packages needed by this notebook when they are missing.
REQUIRED_PACKAGES = {
    "fitz": "pymupdf",
    "numpy": "numpy",
    "pydantic": "pydantic>=2.0",
    "reportlab": "reportlab",
    "sentence_transformers": "sentence-transformers",
}

missing_packages: list[str] = []
for module_name, package_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing_packages.append(package_name)

if missing_packages:
    print("Installing missing packages:", ", ".join(missing_packages))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing_packages]
    )

import fitz
import numpy as np
from pydantic import BaseModel, Field

print(f"Python: {sys.executable}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Module 1/2 output root: {OUTPUT_DIR}")

Python: c:\Users\hp\edtech model testing\.venv\Scripts\python.exe
Project root: C:\Users\hp\EDTECH\Agent_1
Module 1/2 output root: C:\Users\hp\EDTECH\Agent_1\OUTPUT


# 1. Run configuration

The folder named `testing` is the extra thirteenth sample. It is excluded so the final batch processes the requested **12 cleaned transcripts**.

In [2]:
EXPECTED_TRANSCRIPT_COUNT = 12
EXCLUDED_TRANSCRIPT_FOLDERS = {"testing"}

MODULE1_INPUT_PDF = "01_preprocessing.pdf"
MODULE2_OUTPUT_PDF = "02_chunking.pdf"
MODULE2_OUTPUT_JSON = "02_chunking.json"

OVERWRITE_EXISTING_OUTPUTS = True
RUN_EXISTING_MODULE2_TESTS = True
RUN_QWEN_COMPARISON = False
RUN_FULL_QWEN_BATCH = False

# Final production model selected for semantic chunking.
FINAL_CHUNKING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Historical comparison model retained from the existing Module 2 notebook.
QWEN_COMPARISON_MODEL = "Qwen/Qwen3-Embedding-0.6B"

print("Expected cleaned transcripts:", EXPECTED_TRANSCRIPT_COUNT)
print("Excluded folders:", sorted(EXCLUDED_TRANSCRIPT_FOLDERS))
print("Final model:", FINAL_CHUNKING_MODEL)

Expected cleaned transcripts: 12
Excluded folders: ['testing']
Final model: sentence-transformers/all-MiniLM-L6-v2


# 2. Embedded chunk schemas

Copied into the notebook from the existing `app/schemas/chunk.py` implementation.

In [3]:
from __future__ import annotations

from typing import Literal

from pydantic import BaseModel, Field


BoundaryReason = Literal[
    "semantic_shift",
    "transition_phrase",
    "semantic_shift+transition_phrase",
    "max_size",
    "end_of_transcript",
]

TransitionStrength = Literal[
    "strong",
    "soft",
]

SegmentPosition = Literal[
    "single",
    "start",
    "middle",
    "end",
]

ContinuationReason = Literal[
    "max_size_split",
]


class TranscriptChunk(BaseModel):
    """
    One meaningful section of a lesson transcript.

    `chunk_id` identifies the physical chunk.

    `segment_id` identifies the larger logical lesson segment. Multiple
    chunks can belong to the same segment when the chunker is forced to
    split a long, continuous discussion because of max_chunk_words.
    """

    chunk_id: int = Field(ge=1)

    # Actual text passed downstream.
    # For a max-size split, this may include a small overlap from
    # the previous chunk for context preservation.
    text: str = Field(min_length=1)

    word_count: int = Field(ge=1)
    sentence_count: int = Field(ge=1)

    # Sentence range represented by `text`.
    # This may overlap the previous chunk only after a forced max-size split.
    start_sentence: int = Field(ge=0)
    end_sentence: int = Field(ge=0)

    # Non-overlapping/core sentence range belonging to this chunk.
    core_start_sentence: int = Field(ge=0)
    core_end_sentence: int = Field(ge=0)

    # Why this chunk ended.
    boundary_reason: BoundaryReason

    # Similarity between semantic units around the ending boundary.
    boundary_similarity: float | None = Field(
        default=None,
        ge=-1.0,
        le=1.0,
    )

    # Whether a transition phrase supported the ending boundary.
    boundary_transition_strength: TransitionStrength | None = None

    # Context repeated from the previous chunk.
    # Non-zero only when the previous chunk ended because of max_size.
    overlap_word_count: int = Field(
        default=0,
        ge=0,
    )

    # -------------------------------------------------------------
    # Logical segment and continuation metadata
    # -------------------------------------------------------------

    # Human-readable logical segment identifier.
    segment_id: str = Field(
        default="segment_001",
        min_length=1,
    )

    # First physical chunk belonging to this logical segment.
    segment_root_chunk_id: int = Field(
        default=1,
        ge=1,
    )

    # Position of this chunk inside its logical segment.
    segment_chunk_index: int = Field(
        default=1,
        ge=1,
    )

    # Total number of physical chunks in the logical segment.
    segment_chunk_count: int = Field(
        default=1,
        ge=1,
    )

    segment_position: SegmentPosition = "single"

    # True when this chunk continues the same logical segment because
    # the previous physical chunk was forcibly split at max size.
    is_continuation: bool = False

    # Immediate previous physical chunk continued by this chunk.
    continuation_of_chunk_id: int | None = Field(
        default=None,
        ge=1,
    )

    continuation_reason: ContinuationReason | None = None


class ChunkingResult(BaseModel):
    """Complete output of Module 2."""

    chunks: list[TranscriptChunk]

    total_sentences: int = Field(ge=0)
    total_words: int = Field(ge=0)
    semantic_unit_count: int = Field(ge=0)

    # Number of logical lesson segments after grouping forced
    # max-size continuations.
    segment_count: int = Field(
        default=0,
        ge=0,
    )

    embedding_model: str

    # Actual semantic threshold calculated for this transcript.
    semantic_threshold: float

    # Effective configuration for reproducible testing.
    min_chunk_words: int = Field(ge=1)
    target_chunk_words: int = Field(ge=1)
    max_chunk_words: int = Field(ge=1)
    max_size_overlap_words: int = Field(ge=0)

# 3. Embedded embedding service

The production branch retains normalized SentenceTransformer embeddings and model caching. No `.py` service import is required.

In [4]:
from __future__ import annotations

import os
from collections.abc import Sequence
from functools import lru_cache

import numpy as np

# Module 2 and Module 3 use MiniLM in the current project.
DEFAULT_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHUNKING_EMBEDDING_MODEL = DEFAULT_EMBEDDING_MODEL
TOPIC_EMBEDDING_MODEL = DEFAULT_EMBEDDING_MODEL

# Keep this False for the real final workflow. It exists only so the notebook's
# non-model logic can be smoke-tested in an offline environment where the
# sentence-transformers package/model cannot be installed.
ALLOW_OFFLINE_HASHING_FALLBACK = False

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    SentenceTransformer = None


@lru_cache(maxsize=4)
def get_embedding_model(
    model_name: str = DEFAULT_EMBEDDING_MODEL,
):
    """Load and cache the SentenceTransformer model used by Module 2."""

    if SentenceTransformer is None:
        raise ImportError(
            "sentence-transformers is unavailable. Run the environment setup "
            "cell with internet access, or install it from requirements.txt."
        )

    device = os.getenv("EMBEDDING_DEVICE")
    kwargs: dict[str, str] = {}
    if device:
        kwargs["device"] = device

    return SentenceTransformer(model_name, **kwargs)


def _offline_hashing_embeddings(texts: Sequence[str]) -> np.ndarray:
    """Explicitly opt-in fallback for offline notebook validation only."""

    from sklearn.feature_extraction.text import HashingVectorizer
    from sklearn.preprocessing import normalize

    vectorizer = HashingVectorizer(
        n_features=384,
        alternate_sign=False,
        norm=None,
        ngram_range=(1, 2),
    )
    matrix = vectorizer.transform(texts)
    matrix = normalize(matrix, norm="l2", axis=1)
    return matrix.toarray().astype(np.float32)


def embed_texts(
    texts: Sequence[str],
    model_name: str = DEFAULT_EMBEDDING_MODEL,
    batch_size: int = 32,
) -> np.ndarray:
    """
    Generate normalized embeddings for multiple text values.

    The production branch is the existing MiniLM SentenceTransformer logic.
    """

    cleaned_texts = [
        str(text).strip()
        for text in texts
        if str(text).strip()
    ]

    if not cleaned_texts:
        return np.empty((0, 0), dtype=np.float32)

    if batch_size < 1:
        raise ValueError("batch_size must be at least 1.")

    if SentenceTransformer is None:
        if ALLOW_OFFLINE_HASHING_FALLBACK:
            return _offline_hashing_embeddings(cleaned_texts)
        raise ImportError(
            "sentence-transformers is unavailable and the offline test "
            "fallback is disabled."
        )

    model = get_embedding_model(model_name)
    embeddings = model.encode(
        cleaned_texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return np.asarray(embeddings, dtype=np.float32)


def get_embedding_dimension(
    model_name: str = DEFAULT_EMBEDDING_MODEL,
) -> int:
    """Return the model's output vector dimension."""

    if SentenceTransformer is None and ALLOW_OFFLINE_HASHING_FALLBACK:
        return 384

    model = get_embedding_model(model_name)
    dimension = model.get_sentence_embedding_dimension()
    if dimension is None or dimension < 1:
        raise RuntimeError(
            f"Could not determine embedding dimension for {model_name}."
        )
    return int(dimension)

# 4. Embedded semantic chunker

This is the existing Module 2 chunking implementation, including:

- punctuation-aware and punctuationless sentence splitting;
- transition phrase detection;
- semantic-unit construction;
- neighbour cosine similarity;
- adaptive MiniLM thresholding;
- guarded boundary planning;
- target/min/max chunk sizes;
- max-size overlap;
- segment and continuation metadata.

In [5]:
from __future__ import annotations

import re
from collections import defaultdict
from dataclasses import dataclass

import numpy as np

# ChunkingResult and TranscriptChunk are defined in the schema cell above.
# CHUNKING_EMBEDDING_MODEL and embed_texts are defined in the embedding cell above.


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------


@dataclass(frozen=True)
class SemanticChunkingConfig:
    """
    Configuration for guarded semantic chunking.

    Design goals:
    - avoid over-fragmenting teacher/student discussion
    - respect explicit lesson transitions
    - prevent excessively large chunks
    - use overlap only when size forces a split
    - add deterministic logical segment metadata
    - use MiniLM-specific adaptive similarity thresholds
    """

    min_chunk_words: int = 150
    target_chunk_words: int = 325
    max_chunk_words: int = 550

    strong_transition_min_words: int = 80
    semantic_unit_words: int = 60

    # Raw YouTube/ASR captions often contain almost no punctuation.
    # Any punctuation-delimited span longer than this limit is split using
    # conservative discourse/clause boundaries, then by a hard word window.
    max_sentence_words: int = 42
    sentence_split_search_window: int = 12

    boundary_percentile: float = 15.0
    threshold_floor: float = 0.10
    threshold_ceiling: float = 0.45

    soft_transition_margin: float = 0.10
    soft_transition_similarity_ceiling: float = 0.35

    size_penalty_weight: float = 0.12
    strong_transition_bonus: float = 0.10
    soft_transition_bonus: float = 0.04

    max_size_overlap_words: int = 45
    max_size_overlap_sentences: int = 2

    embedding_model: str = CHUNKING_EMBEDDING_MODEL

    def __post_init__(self) -> None:
        if not (
            0
            < self.strong_transition_min_words
            <= self.min_chunk_words
            <= self.target_chunk_words
            <= self.max_chunk_words
        ):
            raise ValueError(
                "Chunk sizes must satisfy: "
                "0 < strong_transition_min <= min <= target <= max."
            )

        if self.semantic_unit_words <= 0:
            raise ValueError(
                "semantic_unit_words must be positive."
            )

        if self.max_sentence_words < 15:
            raise ValueError(
                "max_sentence_words must be at least 15."
            )

        if not (
            1
            <= self.sentence_split_search_window
            < self.max_sentence_words
        ):
            raise ValueError(
                "sentence_split_search_window must be positive and smaller "
                "than max_sentence_words."
            )

        if not 0 <= self.boundary_percentile <= 100:
            raise ValueError(
                "boundary_percentile must be between 0 and 100."
            )

        if not -1.0 <= self.threshold_floor <= 1.0:
            raise ValueError(
                "threshold_floor must be between -1 and 1."
            )

        if not -1.0 <= self.threshold_ceiling <= 1.0:
            raise ValueError(
                "threshold_ceiling must be between -1 and 1."
            )

        if self.threshold_floor > self.threshold_ceiling:
            raise ValueError(
                "threshold_floor cannot be greater than threshold_ceiling."
            )

        if self.soft_transition_margin < 0:
            raise ValueError(
                "soft_transition_margin cannot be negative."
            )

        if not -1.0 <= self.soft_transition_similarity_ceiling <= 1.0:
            raise ValueError(
                "soft_transition_similarity_ceiling must be between -1 and 1."
            )

        if self.max_size_overlap_words < 0:
            raise ValueError(
                "max_size_overlap_words cannot be negative."
            )

        if self.max_size_overlap_sentences < 0:
            raise ValueError(
                "max_size_overlap_sentences cannot be negative."
            )


# ---------------------------------------------------------------------
# Internal structures
# ---------------------------------------------------------------------


@dataclass(frozen=True)
class _SemanticUnit:
    text: str
    start_sentence: int
    end_sentence: int
    word_count: int

    # Transition at the START of this unit.
    transition_strength: str | None = None


@dataclass(frozen=True)
class _Boundary:
    """Candidate boundary occurring AFTER unit_index."""

    unit_index: int
    similarity: float
    reason: str
    transition_strength: str | None = None


@dataclass(frozen=True)
class _ChunkPlan:
    """
    Non-overlapping/core chunk boundaries.

    Overlap and logical segment metadata are added while materializing
    final TranscriptChunk objects.
    """

    start_unit: int
    end_unit: int
    reason: str
    similarity: float | None
    transition_strength: str | None = None


# ---------------------------------------------------------------------
# Semantic Chunker
# ---------------------------------------------------------------------


class SemanticChunker:
    """
    Module 2.

    Input:
        Final cleaned transcript from Module 1.

    Output:
        Meaningful physical chunks plus logical segment metadata.

    A logical segment may contain multiple physical chunks when a long,
    continuous discussion must be split only because of max_chunk_words.

    This module does NOT:
    - identify topics
    - map to the AQA syllabus
    - call an LLM
    - store transcript embeddings
    """

    STRONG_TRANSITION_PATTERNS = (
        r"\bnext (?:chapter|topic|concept|section|question)\b",
        r"\bthe next (?:chapter|topic|concept|section|question)\b",
        r"\bstart (?:the )?(?:next )?chapter\b",
        r"\bbegin (?:the )?(?:next )?chapter\b",
        r"\bchapter (?:number )?\d+\b",
        r"\bnew (?:chapter|topic|concept|section)\b",
        r"\bmove on to (?:the )?(?:next )?(?:chapter|topic|concept|section)\b",
        r"\blet'?s move on to (?:the )?next (?:thing|one|question)\b",
        r"\bmove on to (?:the )?next (?:thing|one|question)\b",
        r"\blet'?s look at (?:the )?next question\b",
        r"\blet'?s go to (?:the )?next question\b",
    )

    SOFT_TRANSITION_PATTERNS = (
        r"\blet'?s move on\b",
        r"\bmove on to\b",
        r"\bmoving on\b",
        r"\blet'?s talk about\b",
        r"\blet'?s discuss\b",
        r"\blet'?s look at\b",
        r"\bnext one\b",
        r"\bnext thing\b",
        r"\bnow (?:we can|we will|let'?s) move\b",
        r"\bnow (?:we can|we will|let'?s) (?:talk|discuss|look)\b",
    )

    def __init__(
        self,
        config: SemanticChunkingConfig | None = None,
    ) -> None:
        self.config = (
            config
            or SemanticChunkingConfig()
        )

    # -----------------------------------------------------------------
    # Public API
    # -----------------------------------------------------------------

    def chunk(
        self,
        cleaned_transcript: str,
    ) -> ChunkingResult:
        """Convert a cleaned transcript into guarded semantic chunks."""

        if not isinstance(
            cleaned_transcript,
            str,
        ):
            raise TypeError(
                "cleaned_transcript must be a string."
            )

        cleaned_transcript = cleaned_transcript.strip()

        if not cleaned_transcript:
            raise ValueError(
                "Cannot chunk an empty transcript."
            )

        sentences = self._split_sentences(
            cleaned_transcript
        )

        if not sentences:
            raise ValueError(
                "No usable sentences found in transcript."
            )

        units = self._build_semantic_units(
            sentences
        )

        if not units:
            raise ValueError(
                "No semantic units could be created."
            )

        if len(units) == 1:
            unit = units[0]

            chunk = TranscriptChunk(
                chunk_id=1,
                text=unit.text,
                word_count=unit.word_count,
                sentence_count=(
                    unit.end_sentence
                    - unit.start_sentence
                    + 1
                ),
                start_sentence=unit.start_sentence,
                end_sentence=unit.end_sentence,
                core_start_sentence=unit.start_sentence,
                core_end_sentence=unit.end_sentence,
                boundary_reason="end_of_transcript",
                boundary_similarity=None,
                boundary_transition_strength=None,
                overlap_word_count=0,
                segment_id="segment_001",
                segment_root_chunk_id=1,
                segment_chunk_index=1,
                segment_chunk_count=1,
                segment_position="single",
                is_continuation=False,
                continuation_of_chunk_id=None,
                continuation_reason=None,
            )

            return self._build_result(
                chunks=[chunk],
                sentences=sentences,
                units=units,
                cleaned_transcript=cleaned_transcript,
                semantic_threshold=self.config.threshold_ceiling,
            )

        unit_embeddings = embed_texts(
            [
                unit.text
                for unit in units
            ],
            model_name=self.config.embedding_model,
        )

        similarities = self._neighbour_similarities(
            unit_embeddings
        )

        semantic_threshold = self._calculate_threshold(
            similarities
        )

        boundaries = self._detect_boundaries(
            units=units,
            similarities=similarities,
            semantic_threshold=semantic_threshold,
        )

        plans = self._build_chunk_plans(
            units=units,
            similarities=similarities,
            boundaries=boundaries,
        )

        chunks = self._materialize_chunks(
            plans=plans,
            units=units,
            sentences=sentences,
        )

        return self._build_result(
            chunks=chunks,
            sentences=sentences,
            units=units,
            cleaned_transcript=cleaned_transcript,
            semantic_threshold=semantic_threshold,
        )

    def _build_result(
        self,
        chunks: list[TranscriptChunk],
        sentences: list[str],
        units: list[_SemanticUnit],
        cleaned_transcript: str,
        semantic_threshold: float,
    ) -> ChunkingResult:
        segment_count = len(
            {
                chunk.segment_id
                for chunk in chunks
            }
        )

        return ChunkingResult(
            chunks=chunks,
            total_sentences=len(sentences),
            total_words=self._word_count(
                cleaned_transcript
            ),
            semantic_unit_count=len(units),
            segment_count=segment_count,
            embedding_model=self.config.embedding_model,
            semantic_threshold=round(
                semantic_threshold,
                4,
            ),
            min_chunk_words=self.config.min_chunk_words,
            target_chunk_words=self.config.target_chunk_words,
            max_chunk_words=self.config.max_chunk_words,
            max_size_overlap_words=(
                self.config.max_size_overlap_words
            ),
        )

    # -----------------------------------------------------------------
    # Sentence preparation
    # -----------------------------------------------------------------

    def _split_sentences(
        self,
        text: str,
    ) -> list[str]:
        """
        Segment ordinary prose and punctuation-poor ASR/YouTube captions.

        Normal punctuated sentences are preserved. Newlines are treated as
        weak boundaries before they are collapsed. Any remaining span that
        is too long is split near a conservative discourse/clause marker.
        When no suitable marker exists, a hard word-window split guarantees
        that one malformed caption block cannot become a 1,000-word
        sentence or semantic unit.

        A period inside ``array.length`` is not split because there is no
        whitespace after that period.
        """

        text = text.replace("\r\n", "\n").replace("\r", "\n")
        text = re.sub(r"[ \t]+", " ", text)

        # Preserve meaningful line boundaries from transcripts while
        # ignoring blank-line noise. A line can still contain several
        # ordinary punctuated sentences.
        lines = [
            line.strip()
            for line in re.split(r"\n+", text)
            if line.strip()
        ]

        initial_parts: list[str] = []

        for line in lines or [text.strip()]:
            initial_parts.extend(
                part.strip()
                for part in re.split(
                    r"(?<=[.!?])\s+",
                    line,
                )
                if part.strip()
            )

        sentences: list[str] = []

        for part in initial_parts:
            sentences.extend(
                self._split_overlong_sentence(part)
            )

        return [
            sentence
            for sentence in sentences
            if sentence
        ]

    def _split_overlong_sentence(
        self,
        sentence: str,
    ) -> list[str]:
        """Split a punctuation-poor span into bounded sentence-like units."""

        words = sentence.split()

        if len(words) <= self.config.max_sentence_words:
            return [sentence.strip()]

        result: list[str] = []
        start = 0

        while start < len(words):
            remaining = len(words) - start

            if remaining <= self.config.max_sentence_words:
                result.append(" ".join(words[start:]).strip())
                break

            target_end = start + self.config.max_sentence_words
            search_start = max(
                start + 12,
                target_end - self.config.sentence_split_search_window,
            )
            # Never search beyond target_end. Otherwise choosing a
            # discourse marker after the hard limit can create a sentence
            # longer than max_sentence_words.
            search_end = min(
                len(words) - 1,
                target_end,
            )

            split_at: int | None = None

            # Prefer boundaries before common spoken-discourse markers.
            # Search backwards so the resulting unit stays near the target.
            for index in range(search_end, search_start - 1, -1):
                token = re.sub(
                    r"^[^a-z0-9]+|[^a-z0-9]+$",
                    "",
                    words[index].lower(),
                )

                previous = (
                    re.sub(
                        r"^[^a-z0-9]+|[^a-z0-9]+$",
                        "",
                        words[index - 1].lower(),
                    )
                    if index > start
                    else ""
                )

                two_word_marker = f"{previous} {token}".strip()

                if (
                    token in {
                        "but",
                        "however",
                        "whereas",
                        "therefore",
                        "because",
                        "although",
                        "though",
                        "meanwhile",
                        "finally",
                        "next",
                        "now",
                        "so",
                        "if",
                        "when",
                        "while",
                        "then",
                    }
                    or two_word_marker in {
                        "for example",
                        "for instance",
                        "on the",
                        "in this",
                        "we can",
                        "we have",
                        "we now",
                        "let us",
                    }
                ):
                    split_at = index
                    break

            if split_at is None or split_at <= start:
                split_at = target_end

            result.append(
                " ".join(words[start:split_at]).strip()
            )
            start = split_at

        return result

    # -----------------------------------------------------------------
    # Transition detection
    # -----------------------------------------------------------------

    def _transition_strength(
        self,
        sentence: str,
    ) -> str | None:
        head = sentence[:220].lower()

        if any(
            re.search(pattern, head)
            for pattern
            in self.STRONG_TRANSITION_PATTERNS
        ):
            return "strong"

        if any(
            re.search(pattern, head)
            for pattern
            in self.SOFT_TRANSITION_PATTERNS
        ):
            return "soft"

        return None

    # -----------------------------------------------------------------
    # Semantic units
    # -----------------------------------------------------------------

    def _build_semantic_units(
        self,
        sentences: list[str],
    ) -> list[_SemanticUnit]:
        units: list[_SemanticUnit] = []

        buffer: list[str] = []
        buffer_words = 0
        start_sentence = 0
        buffer_transition_strength: str | None = None

        def flush_buffer(
            end_sentence: int,
        ) -> None:
            nonlocal buffer
            nonlocal buffer_words
            nonlocal start_sentence
            nonlocal buffer_transition_strength

            if not buffer:
                return

            units.append(
                _SemanticUnit(
                    text=" ".join(buffer),
                    start_sentence=start_sentence,
                    end_sentence=end_sentence,
                    word_count=buffer_words,
                    transition_strength=(
                        buffer_transition_strength
                    ),
                )
            )

            buffer = []
            buffer_words = 0
            buffer_transition_strength = None

        for sentence_index, sentence in enumerate(
            sentences
        ):
            transition_strength = self._transition_strength(
                sentence
            )

            if transition_strength and buffer:
                flush_buffer(
                    sentence_index - 1
                )

            if not buffer:
                start_sentence = sentence_index
                buffer_transition_strength = (
                    transition_strength
                )

            buffer.append(sentence)
            buffer_words += self._word_count(
                sentence
            )

            if (
                buffer_words
                >= self.config.semantic_unit_words
            ):
                flush_buffer(
                    sentence_index
                )

        if buffer:
            flush_buffer(
                len(sentences) - 1
            )

        if len(units) >= 2:
            minimum_tail = max(
                15,
                self.config.semantic_unit_words
                // 2,
            )

            last = units[-1]

            if (
                last.word_count < minimum_tail
                and last.transition_strength is None
            ):
                previous = units[-2]

                units[-2] = _SemanticUnit(
                    text=(
                        previous.text
                        + " "
                        + last.text
                    ),
                    start_sentence=(
                        previous.start_sentence
                    ),
                    end_sentence=(
                        last.end_sentence
                    ),
                    word_count=(
                        previous.word_count
                        + last.word_count
                    ),
                    transition_strength=(
                        previous.transition_strength
                    ),
                )

                units.pop()

        return units

    # -----------------------------------------------------------------
    # Similarity
    # -----------------------------------------------------------------

    @staticmethod
    def _neighbour_similarities(
        embeddings: np.ndarray,
    ) -> np.ndarray:
        if len(embeddings) < 2:
            return np.array(
                [],
                dtype=np.float32,
            )

        similarities = np.sum(
            embeddings[:-1]
            * embeddings[1:],
            axis=1,
        )

        return similarities.astype(
            np.float32
        )

    def _calculate_threshold(
        self,
        similarities: np.ndarray,
    ) -> float:
        if len(similarities) == 0:
            return self.config.threshold_ceiling

        percentile_value = float(
            np.percentile(
                similarities,
                self.config.boundary_percentile,
            )
        )

        threshold = max(
            self.config.threshold_floor,
            min(
                self.config.threshold_ceiling,
                percentile_value,
            ),
        )

        return round(
            threshold,
            4,
        )

    def _soft_transition_threshold(
        self,
        semantic_threshold: float,
    ) -> float:
        return min(
            self.config.soft_transition_similarity_ceiling,
            semantic_threshold
            + self.config.soft_transition_margin,
        )

    # -----------------------------------------------------------------
    # Boundary detection
    # -----------------------------------------------------------------

    def _detect_boundaries(
        self,
        units: list[_SemanticUnit],
        similarities: np.ndarray,
        semantic_threshold: float,
    ) -> dict[int, _Boundary]:
        boundaries: dict[int, _Boundary] = {}

        soft_threshold = self._soft_transition_threshold(
            semantic_threshold
        )

        for index, similarity_value in enumerate(
            similarities
        ):
            similarity = float(
                similarity_value
            )

            semantic_shift = (
                similarity
                <= semantic_threshold
            )

            next_unit = units[index + 1]

            transition_strength = (
                next_unit.transition_strength
            )

            strong_transition = (
                transition_strength == "strong"
            )

            soft_transition = (
                transition_strength == "soft"
                and similarity <= soft_threshold
            )

            transition_boundary = (
                strong_transition
                or soft_transition
            )

            if not (
                semantic_shift
                or transition_boundary
            ):
                continue

            if (
                semantic_shift
                and transition_boundary
            ):
                reason = (
                    "semantic_shift+transition_phrase"
                )
            elif transition_boundary:
                reason = "transition_phrase"
            else:
                reason = "semantic_shift"

            boundaries[index] = _Boundary(
                unit_index=index,
                similarity=similarity,
                reason=reason,
                transition_strength=(
                    transition_strength
                    if transition_boundary
                    else None
                ),
            )

        return boundaries

    # -----------------------------------------------------------------
    # Chunk planning
    # -----------------------------------------------------------------

    def _build_chunk_plans(
        self,
        units: list[_SemanticUnit],
        similarities: np.ndarray,
        boundaries: dict[int, _Boundary],
    ) -> list[_ChunkPlan]:
        prefix_words = [0]

        for unit in units:
            prefix_words.append(
                prefix_words[-1]
                + unit.word_count
            )

        def words_between(
            start: int,
            end: int,
        ) -> int:
            return (
                prefix_words[end + 1]
                - prefix_words[start]
            )

        plans: list[_ChunkPlan] = []
        start = 0
        number_of_units = len(units)

        while start < number_of_units:
            remaining_words = words_between(
                start,
                number_of_units - 1,
            )

            candidates: list[
                tuple[int, _Boundary]
            ] = []

            for end_index in range(
                start,
                number_of_units - 1,
            ):
                current_words = words_between(
                    start,
                    end_index,
                )

                if (
                    current_words
                    > self.config.max_chunk_words
                ):
                    break

                boundary = boundaries.get(
                    end_index
                )

                if boundary is None:
                    continue

                required_min = (
                    self.config.strong_transition_min_words
                    if (
                        boundary.transition_strength
                        == "strong"
                    )
                    else self.config.min_chunk_words
                )

                if current_words < required_min:
                    continue

                words_after = words_between(
                    end_index + 1,
                    number_of_units - 1,
                )

                if (
                    0
                    < words_after
                    < self.config.min_chunk_words
                ):
                    continue

                candidates.append(
                    (
                        end_index,
                        boundary,
                    )
                )

            if candidates:
                def candidate_score(
                    candidate: tuple[
                        int,
                        _Boundary,
                    ],
                ) -> float:
                    end_index, boundary = candidate

                    current_words = words_between(
                        start,
                        end_index,
                    )

                    size_distance = abs(
                        current_words
                        - self.config.target_chunk_words
                    )

                    size_penalty = (
                        size_distance
                        / self.config.target_chunk_words
                    )

                    score = (
                        boundary.similarity
                        + (
                            self.config.size_penalty_weight
                            * size_penalty
                        )
                    )

                    if (
                        boundary.transition_strength
                        == "strong"
                    ):
                        score -= (
                            self.config.strong_transition_bonus
                        )
                    elif (
                        boundary.transition_strength
                        == "soft"
                    ):
                        score -= (
                            self.config.soft_transition_bonus
                        )

                    return score

                end, selected = min(
                    candidates,
                    key=candidate_score,
                )

                plans.append(
                    _ChunkPlan(
                        start_unit=start,
                        end_unit=end,
                        reason=selected.reason,
                        similarity=selected.similarity,
                        transition_strength=(
                            selected.transition_strength
                        ),
                    )
                )

                start = end + 1
                continue

            if (
                remaining_words
                <= self.config.max_chunk_words
            ):
                plans.append(
                    _ChunkPlan(
                        start_unit=start,
                        end_unit=(
                            number_of_units - 1
                        ),
                        reason="end_of_transcript",
                        similarity=None,
                        transition_strength=None,
                    )
                )
                break

            possible_ends: list[int] = []

            for end_index in range(
                start,
                number_of_units - 1,
            ):
                current_words = words_between(
                    start,
                    end_index,
                )

                if (
                    current_words
                    > self.config.max_chunk_words
                ):
                    break

                words_after = words_between(
                    end_index + 1,
                    number_of_units - 1,
                )

                if (
                    words_after
                    >= self.config.min_chunk_words
                ):
                    possible_ends.append(
                        end_index
                    )

            if possible_ends:
                end = possible_ends[-1]
            else:
                end = start

            if end < len(similarities):
                similarity: float | None = float(
                    similarities[end]
                )
            else:
                similarity = None

            plans.append(
                _ChunkPlan(
                    start_unit=start,
                    end_unit=end,
                    reason="max_size",
                    similarity=similarity,
                    transition_strength=None,
                )
            )

            start = end + 1

        return plans

    # -----------------------------------------------------------------
    # Materialize chunks + logical segment metadata
    # -----------------------------------------------------------------

    def _materialize_chunks(
        self,
        plans: list[_ChunkPlan],
        units: list[_SemanticUnit],
        sentences: list[str],
    ) -> list[TranscriptChunk]:
        chunks: list[TranscriptChunk] = []

        for index, plan in enumerate(
            plans
        ):
            core_start_sentence = (
                units[
                    plan.start_unit
                ].start_sentence
            )

            core_end_sentence = (
                units[
                    plan.end_unit
                ].end_sentence
            )

            text_start_sentence = (
                core_start_sentence
            )

            overlap_word_count = 0

            if (
                index > 0
                and plans[
                    index - 1
                ].reason == "max_size"
            ):
                (
                    text_start_sentence,
                    overlap_word_count,
                ) = self._find_overlap_start(
                    sentences=sentences,
                    core_start_sentence=(
                        core_start_sentence
                    ),
                )

            selected_sentences = sentences[
                text_start_sentence
                : core_end_sentence + 1
            ]

            text = " ".join(
                selected_sentences
            ).strip()

            chunks.append(
                TranscriptChunk(
                    chunk_id=index + 1,
                    text=text,
                    word_count=self._word_count(
                        text
                    ),
                    sentence_count=(
                        core_end_sentence
                        - text_start_sentence
                        + 1
                    ),
                    start_sentence=(
                        text_start_sentence
                    ),
                    end_sentence=(
                        core_end_sentence
                    ),
                    core_start_sentence=(
                        core_start_sentence
                    ),
                    core_end_sentence=(
                        core_end_sentence
                    ),
                    boundary_reason=(
                        plan.reason
                    ),
                    boundary_similarity=(
                        round(
                            plan.similarity,
                            4,
                        )
                        if (
                            plan.similarity
                            is not None
                        )
                        else None
                    ),
                    boundary_transition_strength=(
                        plan.transition_strength
                    ),
                    overlap_word_count=(
                        overlap_word_count
                    ),
                )
            )

        return self._assign_segment_metadata(
            chunks=chunks,
            plans=plans,
        )

    def _assign_segment_metadata(
        self,
        *,
        chunks: list[TranscriptChunk],
        plans: list[_ChunkPlan],
    ) -> list[TranscriptChunk]:
        """
        Group physical chunks into deterministic logical segments.

        Safe rule:
        - A chunk continues the same segment only when the previous
          chunk ended because of `max_size`.
        - Semantic and transition boundaries start a new segment.

        This avoids guessing topic identity inside Module 2 while still
        telling Module 3 which chunks are definitely continuations.
        """

        if not chunks:
            return []

        assignments: list[dict[str, object]] = []

        segment_number = 1
        segment_root_chunk_id = chunks[0].chunk_id
        segment_chunk_index = 1

        assignments.append(
            {
                "segment_number": segment_number,
                "segment_root_chunk_id": (
                    segment_root_chunk_id
                ),
                "segment_chunk_index": (
                    segment_chunk_index
                ),
                "is_continuation": False,
                "continuation_of_chunk_id": None,
                "continuation_reason": None,
            }
        )

        for index in range(
            1,
            len(chunks),
        ):
            previous_plan = plans[
                index - 1
            ]

            is_continuation = (
                previous_plan.reason
                == "max_size"
            )

            if is_continuation:
                segment_chunk_index += 1
            else:
                segment_number += 1
                segment_root_chunk_id = (
                    chunks[index].chunk_id
                )
                segment_chunk_index = 1

            assignments.append(
                {
                    "segment_number": segment_number,
                    "segment_root_chunk_id": (
                        segment_root_chunk_id
                    ),
                    "segment_chunk_index": (
                        segment_chunk_index
                    ),
                    "is_continuation": (
                        is_continuation
                    ),
                    "continuation_of_chunk_id": (
                        chunks[index - 1].chunk_id
                        if is_continuation
                        else None
                    ),
                    "continuation_reason": (
                        "max_size_split"
                        if is_continuation
                        else None
                    ),
                }
            )

        counts: dict[int, int] = defaultdict(int)

        for assignment in assignments:
            counts[
                int(
                    assignment[
                        "segment_number"
                    ]
                )
            ] += 1

        updated_chunks: list[
            TranscriptChunk
        ] = []

        for chunk, assignment in zip(
            chunks,
            assignments,
            strict=True,
        ):
            segment_number = int(
                assignment[
                    "segment_number"
                ]
            )

            segment_chunk_count = counts[
                segment_number
            ]

            segment_chunk_index = int(
                assignment[
                    "segment_chunk_index"
                ]
            )

            if segment_chunk_count == 1:
                segment_position = "single"
            elif segment_chunk_index == 1:
                segment_position = "start"
            elif (
                segment_chunk_index
                == segment_chunk_count
            ):
                segment_position = "end"
            else:
                segment_position = "middle"

            updates = {
                "segment_id": (
                    f"segment_{segment_number:03d}"
                ),
                "segment_root_chunk_id": int(
                    assignment[
                        "segment_root_chunk_id"
                    ]
                ),
                "segment_chunk_index": (
                    segment_chunk_index
                ),
                "segment_chunk_count": (
                    segment_chunk_count
                ),
                "segment_position": (
                    segment_position
                ),
                "is_continuation": bool(
                    assignment[
                        "is_continuation"
                    ]
                ),
                "continuation_of_chunk_id": (
                    assignment[
                        "continuation_of_chunk_id"
                    ]
                ),
                "continuation_reason": (
                    assignment[
                        "continuation_reason"
                    ]
                ),
            }

            if hasattr(
                chunk,
                "model_copy",
            ):
                updated = chunk.model_copy(
                    update=updates
                )
            else:
                updated = chunk.copy(
                    update=updates
                )

            updated_chunks.append(
                updated
            )

        return updated_chunks

    def _find_overlap_start(
        self,
        sentences: list[str],
        core_start_sentence: int,
    ) -> tuple[int, int]:
        if (
            self.config.max_size_overlap_words
            <= 0
            or self.config.max_size_overlap_sentences
            <= 0
            or core_start_sentence <= 0
        ):
            return (
                core_start_sentence,
                0,
            )

        overlap_start = (
            core_start_sentence
        )

        overlap_words = 0
        overlap_sentences = 0

        sentence_index = (
            core_start_sentence - 1
        )

        while (
            sentence_index >= 0
            and overlap_sentences
            < self.config.max_size_overlap_sentences
        ):
            sentence_words = self._word_count(
                sentences[
                    sentence_index
                ]
            )

            if (
                overlap_sentences > 0
                and (
                    overlap_words
                    + sentence_words
                )
                > self.config.max_size_overlap_words
            ):
                break

            overlap_start = sentence_index
            overlap_words += sentence_words
            overlap_sentences += 1
            sentence_index -= 1

            if (
                overlap_words
                >= self.config.max_size_overlap_words
            ):
                break

        return (
            overlap_start,
            overlap_words,
        )

    # -----------------------------------------------------------------
    # Utility
    # -----------------------------------------------------------------

    @staticmethod
    def _word_count(
        text: str,
    ) -> int:
        return len(
            re.findall(
                r"\S+",
                text,
            )
        )

# 5. Read the final cleaned transcript from Module 1 PDF

Only the section after `Final Cleaned and Technically Normalised Transcript` is extracted. Module 1 summary tables and page labels are not passed into chunking.

In [6]:
MODULE1_FINAL_TEXT_MARKER = (
    "Final Cleaned and Technically Normalised Transcript"
)


def extract_final_cleaned_text_from_pdf(pdf_path: Path) -> str:
    if not pdf_path.exists():
        raise FileNotFoundError(f"Module 1 PDF not found: {pdf_path}")

    with fitz.open(pdf_path) as document:
        raw_text = "\n".join(page.get_text("text") for page in document)

    marker_index = raw_text.find(MODULE1_FINAL_TEXT_MARKER)
    if marker_index < 0:
        raise ValueError(
            f"Final cleaned transcript marker was not found in {pdf_path}. "
            "Regenerate this file with the self-contained Module 1 notebook."
        )

    cleaned_section = raw_text[
        marker_index + len(MODULE1_FINAL_TEXT_MARKER):
    ]

    # Remove the running page labels generated by the Module 1 PDF.
    cleaned_section = re.sub(
        r"(?m)^\s*Page\s+\d+\s*$",
        " ",
        cleaned_section,
    )

    # PDF extraction wraps lines visually. Convert those wraps back into
    # ordinary spaces without changing the words.
    cleaned_section = re.sub(r"\s+", " ", cleaned_section).strip()

    if not cleaned_section:
        raise ValueError(f"No final cleaned transcript found in {pdf_path}")

    return cleaned_section


def discover_module1_outputs() -> list[Path]:
    paths = sorted(
        path
        for path in OUTPUT_DIR.glob(f"*/{MODULE1_INPUT_PDF}")
        if path.parent.name not in EXCLUDED_TRANSCRIPT_FOLDERS
    )

    if len(paths) != EXPECTED_TRANSCRIPT_COUNT:
        discovered = "\n".join(
            f"- {path.parent.name}" for path in paths
        ) or "- none"
        raise AssertionError(
            f"Expected {EXPECTED_TRANSCRIPT_COUNT} Module 1 PDFs but found "
            f"{len(paths)}.\nDiscovered:\n{discovered}\n"
            "Adjust EXCLUDED_TRANSCRIPT_FOLDERS only if your dataset changed."
        )

    return paths


module1_pdf_paths = discover_module1_outputs()

print("Module 1 cleaned PDFs selected for Module 2:")
for index, path in enumerate(module1_pdf_paths, start=1):
    text = extract_final_cleaned_text_from_pdf(path)
    print(
        f"{index:02d}. {path.parent.name:<50} "
        f"{len(text.split()):>6} words"
    )

AssertionError: Expected 12 Module 1 PDFs but found 13.
Discovered:
- 123
- 456
- Real_AQA_02_Number_Bases
- Real_AQA_03_Bubble_Sort
- Real_AQA_Linear_Search
- Transcript
- Transcript 1
- Transcript 2
- Transcript_AQA_Topics
- Transcript_No_CS_Conversation
- Transcript_Raw_2_Networks_Cybersecurity
- Transcript_Raw_3_Algorithms_Programming
- Transcript_Test_1_Data_Representation
Adjust EXCLUDED_TRANSCRIPT_FOLDERS only if your dataset changed.

In [7]:
# ================================================================
# MODULE 2 — SELECT ONLY ONE MODULE 1 OUTPUT
#
# Input:
#   OUTPUT/Transcript/01_preprocessing.pdf
#
# This replaces the original 12-transcript discovery cell.
# ================================================================

from pathlib import Path


PROJECT_ROOT = Path(
    r"C:\Users\hp\EDTECH\Agent_1"
)

OUTPUT_DIR = PROJECT_ROOT / "OUTPUT"

TARGET_TRANSCRIPT_NAME = "Transcript"

target_module1_pdf = (
    OUTPUT_DIR
    / TARGET_TRANSCRIPT_NAME
    / "01_preprocessing.pdf"
)


if not OUTPUT_DIR.is_dir():
    raise FileNotFoundError(
        f"OUTPUT directory was not found:\n{OUTPUT_DIR}"
    )


if not target_module1_pdf.is_file():
    raise FileNotFoundError(
        "Module 1 output for the selected transcript was not found.\n\n"
        f"Expected file:\n{target_module1_pdf}\n\n"
        "Run the Module 1 single-transcript cell first."
    )


if target_module1_pdf.stat().st_size == 0:
    raise ValueError(
        f"Module 1 PDF exists but is empty:\n{target_module1_pdf}"
    )


# Keep the same variable name expected by later notebook cells.
module1_pdf_paths = [
    target_module1_pdf
]


print("=" * 100)
print("MODULE 2 — SINGLE TRANSCRIPT INPUT SELECTED")
print("=" * 100)

print("Project root:")
print(PROJECT_ROOT)

print("\nSelected transcript:")
print(TARGET_TRANSCRIPT_NAME)

print("\nSelected Module 1 PDF:")
print(target_module1_pdf)

print(
    "\nFile size:",
    f"{target_module1_pdf.stat().st_size:,} bytes",
)

print(
    "\nNumber of Module 1 PDFs selected:",
    len(module1_pdf_paths),
)

assert len(module1_pdf_paths) == 1

print("\nReady for single-transcript chunking.")

MODULE 2 — SINGLE TRANSCRIPT INPUT SELECTED
Project root:
C:\Users\hp\EDTECH\Agent_1

Selected transcript:
Transcript

Selected Module 1 PDF:
C:\Users\hp\EDTECH\Agent_1\OUTPUT\Transcript\01_preprocessing.pdf

File size: 3,517 bytes

Number of Module 1 PDFs selected: 1

Ready for single-transcript chunking.


# 6. Existing Module 2 tests migrated into the notebook

These tests preserve the existing transition, punctuationless-caption, semantic-unit, and logical continuation checks. They run without importing any test `.py` file.

In [9]:
def _make_test_chunk(chunk_id: int) -> TranscriptChunk:
    return TranscriptChunk(
        chunk_id=chunk_id,
        text=f"Chunk {chunk_id}",
        word_count=2,
        sentence_count=1,
        start_sentence=chunk_id - 1,
        end_sentence=chunk_id - 1,
        core_start_sentence=chunk_id - 1,
        core_end_sentence=chunk_id - 1,
        boundary_reason="end_of_transcript",
        boundary_similarity=None,
        boundary_transition_strength=None,
        overlap_word_count=0,
    )


def test_before_we_move_on_is_not_a_transition() -> None:
    chunker = SemanticChunker()
    assert chunker._transition_strength(
        "Before we move on, one last question about linear search."
    ) is None


def test_real_transition_still_detected() -> None:
    chunker = SemanticChunker()
    assert chunker._transition_strength(
        "Let's move on to the next question."
    ) == "strong"


def test_punctuationless_caption_is_split_into_bounded_sentences() -> None:
    chunker = SemanticChunker()
    raw_caption = " ".join(
        [
            "algorithm efficiency compares two solutions that perform the same task",
            "the first solution repeats an instruction inside a loop",
            "the second solution calculates the answer using one expression",
            "when the input becomes larger the loop executes many more times",
            "however the direct calculation still executes once",
            "therefore the second algorithm is more time efficient",
        ]
        * 8
    )
    sentences = chunker._split_sentences(raw_caption)
    assert len(sentences) > 1
    assert max(len(sentence.split()) for sentence in sentences) <= (
        chunker.config.max_sentence_words
    )


def test_punctuationless_caption_does_not_create_giant_semantic_unit() -> None:
    chunker = SemanticChunker()
    raw_caption = " ".join(
        [
            "we compare algorithm efficiency using two functions",
            "one function uses a for loop and executes repeatedly",
            "the other function uses one arithmetic expression",
            "as the input size grows the repeated solution takes longer",
            "the direct calculation remains quick",
        ]
        * 25
    )
    sentences = chunker._split_sentences(raw_caption)
    units = chunker._build_semantic_units(sentences)
    assert len(units) > 2
    assert max(unit.word_count for unit in units) < 150


def test_normal_punctuated_sentences_are_preserved() -> None:
    chunker = SemanticChunker()
    text = (
        "Linear search checks each item in order. "
        "Binary search checks the middle item of sorted data. "
        "Both algorithms solve a searching problem."
    )
    assert chunker._split_sentences(text) == [
        "Linear search checks each item in order.",
        "Binary search checks the middle item of sorted data.",
        "Both algorithms solve a searching problem.",
    ]


def test_module2_segment_metadata() -> None:
    chunker = SemanticChunker()
    chunks = [_make_test_chunk(index) for index in range(1, 6)]
    plans = [
        _ChunkPlan(0, 0, "semantic_shift", 0.1),
        _ChunkPlan(1, 1, "max_size", 0.8),
        _ChunkPlan(2, 2, "max_size", 0.8),
        _ChunkPlan(3, 3, "semantic_shift", 0.1),
        _ChunkPlan(4, 4, "end_of_transcript", None),
    ]

    updated = chunker._assign_segment_metadata(chunks=chunks, plans=plans)

    assert updated[0].segment_id == "segment_001"
    assert updated[0].segment_position == "single"
    assert updated[0].is_continuation is False

    assert updated[1].segment_id == "segment_002"
    assert updated[1].segment_position == "start"
    assert updated[1].segment_chunk_count == 3

    assert updated[2].segment_id == "segment_002"
    assert updated[2].segment_position == "middle"
    assert updated[2].is_continuation is True
    assert updated[2].continuation_of_chunk_id == 2
    assert updated[2].segment_root_chunk_id == 2

    assert updated[3].segment_id == "segment_002"
    assert updated[3].segment_position == "end"
    assert updated[3].continuation_of_chunk_id == 3

    assert updated[4].segment_id == "segment_003"
    assert updated[4].segment_position == "single"
    assert updated[4].is_continuation is False


def run_existing_module2_tests() -> None:
    tests = [
        test_before_we_move_on_is_not_a_transition,
        test_real_transition_still_detected,
        test_punctuationless_caption_is_split_into_bounded_sentences,
        test_punctuationless_caption_does_not_create_giant_semantic_unit,
        test_normal_punctuated_sentences_are_preserved,
        test_module2_segment_metadata,
    ]

    for test in tests:
        test()
        print(f"PASS: {test.__name__}")

    print("\nALL MIGRATED MODULE 2 REGRESSION TESTS PASSED")


if RUN_EXISTING_MODULE2_TESTS:
    run_existing_module2_tests()
else:
    print("Existing Module 2 tests retained but skipped by configuration.")

PASS: test_before_we_move_on_is_not_a_transition
PASS: test_real_transition_still_detected
PASS: test_punctuationless_caption_is_split_into_bounded_sentences
PASS: test_punctuationless_caption_does_not_create_giant_semantic_unit
PASS: test_normal_punctuated_sentences_are_preserved
PASS: test_module2_segment_metadata

ALL MIGRATED MODULE 2 REGRESSION TESTS PASSED


# 7. Final MiniLM chunker configuration

The settings below match the existing production test script and semantic chunker defaults.

In [ ]:
FINAL_CONFIG = SemanticChunkingConfig(
    min_chunk_words=150,
    target_chunk_words=325,
    max_chunk_words=550,
    strong_transition_min_words=80,
    semantic_unit_words=60,
    max_sentence_words=42,
    sentence_split_search_window=12,
    boundary_percentile=15.0,
    threshold_floor=0.10,
    threshold_ceiling=0.45,
    soft_transition_margin=0.10,
    soft_transition_similarity_ceiling=0.35,
    max_size_overlap_words=45,
    max_size_overlap_sentences=2,
    embedding_model=FINAL_CHUNKING_MODEL,
)

final_chunker = SemanticChunker(config=FINAL_CONFIG)
print(FINAL_CONFIG)

SemanticChunkingConfig(min_chunk_words=150, target_chunk_words=325, max_chunk_words=550, strong_transition_min_words=80, semantic_unit_words=60, max_sentence_words=42, sentence_split_search_window=12, boundary_percentile=15.0, threshold_floor=0.1, threshold_ceiling=0.45, soft_transition_margin=0.1, soft_transition_similarity_ceiling=0.35, size_penalty_weight=0.12, strong_transition_bonus=0.1, soft_transition_bonus=0.04, max_size_overlap_words=45, max_size_overlap_sentences=2, embedding_model='sentence-transformers/all-MiniLM-L6-v2')


# 8. Existing single-transcript semantic chunking test

This is the notebook form of `scripts/test_semantic_chunking.py`: same configuration, timing, chunk metadata, and real cleaned input.

In [18]:
def result_to_dict(result: ChunkingResult) -> dict[str, Any]:
    if hasattr(result, "model_dump"):
        return result.model_dump()
    return result.dict()


def result_metrics(result: ChunkingResult) -> dict[str, Any]:
    counts = [chunk.word_count for chunk in result.chunks]
    return {
        "embedding_model": result.embedding_model,
        "total_words": result.total_words,
        "total_sentences": result.total_sentences,
        "semantic_units": result.semantic_unit_count,
        "semantic_threshold": result.semantic_threshold,
        "physical_chunks": len(result.chunks),
        "logical_segments": result.segment_count,
        "continuation_chunks": sum(chunk.is_continuation for chunk in result.chunks),
        "overlap_chunks": sum(chunk.overlap_word_count > 0 for chunk in result.chunks),
        "minimum_chunk_words": min(counts),
        "maximum_chunk_words": max(counts),
        "average_chunk_words": round(float(np.mean(counts)), 2),
        "boundary_reasons": dict(Counter(chunk.boundary_reason for chunk in result.chunks)),
    }


PRIMARY_TRANSCRIPT_PDF = module1_pdf_paths[0]
primary_cleaned_text = extract_final_cleaned_text_from_pdf(PRIMARY_TRANSCRIPT_PDF)

single_start = time.perf_counter()
single_result = final_chunker.chunk(primary_cleaned_text)
single_runtime = time.perf_counter() - single_start

print("=" * 100)
print("MODULE 2 - SINGLE CLEANED TRANSCRIPT TEST")
print("=" * 100)
print("Transcript:", PRIMARY_TRANSCRIPT_PDF.parent.name)
print("Runtime seconds:", round(single_runtime, 4))
print(json.dumps(result_metrics(single_result), indent=2, ensure_ascii=False))

for chunk in single_result.chunks:
    print("\n" + "-" * 100)
    print(f"CHUNK {chunk.chunk_id} | {chunk.segment_id} | {chunk.segment_position}")
    print(
        f"Boundary={chunk.boundary_reason} | Words={chunk.word_count} | "
        f"Continuation={chunk.is_continuation} | Overlap={chunk.overlap_word_count}"
    )
    print("-" * 100)
    print(chunk.text)

MODULE 2 - SINGLE CLEANED TRANSCRIPT TEST
Transcript: Transcript
Runtime seconds: 0.048
{
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "total_words": 100,
  "total_sentences": 5,
  "semantic_units": 2,
  "semantic_threshold": 0.45,
  "physical_chunks": 1,
  "logical_segments": 1,
  "continuation_chunks": 0,
  "overlap_chunks": 0,
  "minimum_chunk_words": 100,
  "maximum_chunk_words": 100,
  "average_chunk_words": 100.0,
  "boundary_reasons": {
    "end_of_transcript": 1
  }
}

----------------------------------------------------------------------------------------------------
CHUNK 1 | segment_001 | single
Boundary=end_of_transcript | Words=100 | Continuation=False | Overlap=0
----------------------------------------------------------------------------------------------------
Right, so today we are covering computer networks and cyber security, starting with how devices share data and resources. We compared PANs, LANs, and WANs, and also looked at wired and wireless

# 9. Optional historical MiniLM vs Qwen comparison

This section is retained from the earlier Module 2 notebook. It changes **only** the embedding model. The final production batch below always uses MiniLM unless you intentionally change the configuration.

In [19]:
model_comparison = None

if RUN_QWEN_COMPARISON:
    qwen_config = SemanticChunkingConfig(
        **{
            **FINAL_CONFIG.__dict__,
            "embedding_model": QWEN_COMPARISON_MODEL,
        }
    )
    qwen_chunker = SemanticChunker(config=qwen_config)

    qwen_start = time.perf_counter()
    qwen_result = qwen_chunker.chunk(primary_cleaned_text)
    qwen_runtime = time.perf_counter() - qwen_start

    minilm_boundaries = {
        chunk.core_end_sentence
        for chunk in single_result.chunks
        if chunk.boundary_reason != "end_of_transcript"
    }
    qwen_boundaries = {
        chunk.core_end_sentence
        for chunk in qwen_result.chunks
        if chunk.boundary_reason != "end_of_transcript"
    }
    union = minilm_boundaries | qwen_boundaries
    agreement = (
        len(minilm_boundaries & qwen_boundaries) / len(union)
        if union else 1.0
    )

    model_comparison = {
        "transcript": PRIMARY_TRANSCRIPT_PDF.parent.name,
        "minilm": {
            **result_metrics(single_result),
            "runtime_seconds": round(single_runtime, 4),
        },
        "qwen": {
            **result_metrics(qwen_result),
            "runtime_seconds": round(qwen_runtime, 4),
        },
        "boundary_agreement": round(agreement, 4),
    }
    print(json.dumps(model_comparison, indent=2, ensure_ascii=False))
else:
    print(
        "Historical Qwen comparison retained but skipped. "
        "Set RUN_QWEN_COMPARISON = True to run it."
    )

Historical Qwen comparison retained but skipped. Set RUN_QWEN_COMPARISON = True to run it.


# 10. PDF and JSON output helpers

The PDF is the readable deliverable. The JSON preserves exact structured chunks for Module 3 without having to parse the PDF again.

In [20]:
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import mm
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.platypus import (
    PageBreak,
    Paragraph,
    SimpleDocTemplate,
    Spacer,
    Table,
    TableStyle,
)


def register_readable_font() -> str:
    candidates = [
        Path("C:/Windows/Fonts/arial.ttf"),
        Path("C:/Windows/Fonts/calibri.ttf"),
        Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"),
        Path("/Library/Fonts/Arial.ttf"),
    ]

    for font_path in candidates:
        if font_path.exists():
            try:
                pdfmetrics.registerFont(TTFont("Module2Body", str(font_path)))
                return "Module2Body"
            except Exception:
                continue

    return "Helvetica"


PDF_FONT = register_readable_font()


def _page_header_footer(canvas, document) -> None:
    canvas.saveState()
    canvas.setFont(PDF_FONT, 8)
    canvas.setFillColor(colors.black)
    canvas.drawString(18 * mm, 10 * mm, "Agent 1 - Module 2 Semantic Chunking")
    canvas.drawRightString(
        A4[0] - 18 * mm,
        10 * mm,
        f"Page {document.page}",
    )
    canvas.restoreState()


def export_chunking_pdf(
    *,
    transcript_name: str,
    source_pdf: Path,
    result: ChunkingResult,
    runtime_seconds: float,
    output_path: Path,
) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)

    document = SimpleDocTemplate(
        str(output_path),
        pagesize=A4,
        rightMargin=18 * mm,
        leftMargin=18 * mm,
        topMargin=18 * mm,
        bottomMargin=18 * mm,
        title=f"Module 2 Chunking - {transcript_name}",
        author="Agent 1",
    )

    styles = getSampleStyleSheet()
    title_style = ParagraphStyle(
        "Module2Title",
        parent=styles["Title"],
        fontName=PDF_FONT,
        fontSize=18,
        leading=22,
        alignment=TA_CENTER,
        textColor=colors.black,
        spaceAfter=10,
    )
    heading_style = ParagraphStyle(
        "Module2Heading",
        parent=styles["Heading2"],
        fontName=PDF_FONT,
        fontSize=12,
        leading=15,
        textColor=colors.black,
        spaceBefore=8,
        spaceAfter=6,
    )
    body_style = ParagraphStyle(
        "Module2BodyStyle",
        parent=styles["BodyText"],
        fontName=PDF_FONT,
        fontSize=9.5,
        leading=14,
        alignment=TA_LEFT,
        textColor=colors.black,
        spaceAfter=6,
    )
    small_style = ParagraphStyle(
        "Module2Small",
        parent=body_style,
        fontSize=8.5,
        leading=11,
        splitLongWords=True,
        wordWrap="CJK",
    )

    def pdf_value(value: Any) -> Paragraph:
        if isinstance(value, dict):
            text = "; ".join(
                f"{str(key).replace('_', ' ')}: {item}"
                for key, item in value.items()
            )
        elif isinstance(value, (list, tuple)):
            text = "; ".join(str(item) for item in value)
        elif value is None:
            text = "-"
        else:
            text = str(value)

        # Make technical identifiers readable and safely wrappable in tables.
        text = text.replace("_", " ")
        return Paragraph(escape(text), small_style)

    story = [
        Paragraph("Agent 1 - Module 2 Semantic Chunking", title_style),
        Paragraph(f"<b>Transcript:</b> {escape(transcript_name)}", body_style),
        Paragraph(f"<b>Input:</b> {escape(source_pdf.name)}", body_style),
        Paragraph(
            f"<b>Generated:</b> {datetime.now(timezone.utc).isoformat()}",
            body_style,
        ),
        Spacer(1, 5),
        Paragraph("Chunking Summary", heading_style),
    ]

    summary = result_metrics(result)
    summary_rows = [[pdf_value("Metric"), pdf_value("Value")]] + [
        [pdf_value(str(key).replace("_", " ").title()), pdf_value(value)]
        for key, value in summary.items()
    ] + [[pdf_value("Runtime Seconds"), pdf_value(f"{runtime_seconds:.4f}")]]

    summary_table = Table(summary_rows, colWidths=[67 * mm, 92 * mm], repeatRows=1)
    summary_table.setStyle(
        TableStyle(
            [
                ("FONTNAME", (0, 0), (-1, -1), PDF_FONT),
                ("FONTSIZE", (0, 0), (-1, -1), 8.5),
                ("TEXTCOLOR", (0, 0), (-1, -1), colors.black),
                ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#EAEAEA")),
                ("GRID", (0, 0), (-1, -1), 0.5, colors.black),
                ("VALIGN", (0, 0), (-1, -1), "TOP"),
                ("LEFTPADDING", (0, 0), (-1, -1), 5),
                ("RIGHTPADDING", (0, 0), (-1, -1), 5),
                ("TOPPADDING", (0, 0), (-1, -1), 4),
                ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
            ]
        )
    )
    story.extend([summary_table, PageBreak()])

    for index, chunk in enumerate(result.chunks):
        story.append(
            Paragraph(
                f"Chunk {chunk.chunk_id} - {escape(chunk.segment_id)}",
                heading_style,
            )
        )

        metadata_rows = [
            [pdf_value("Words"), pdf_value(chunk.word_count), pdf_value("Sentences"), pdf_value(chunk.sentence_count)],
            [pdf_value("Boundary"), pdf_value(chunk.boundary_reason), pdf_value("Similarity"), pdf_value(chunk.boundary_similarity)],
            [pdf_value("Segment position"), pdf_value(chunk.segment_position), pdf_value("Segment chunk"), pdf_value(f"{chunk.segment_chunk_index}/{chunk.segment_chunk_count}")],
            [pdf_value("Continuation"), pdf_value(chunk.is_continuation), pdf_value("Continuation of"), pdf_value(chunk.continuation_of_chunk_id)],
            [pdf_value("Overlap words"), pdf_value(chunk.overlap_word_count), pdf_value("Core sentences"), pdf_value(f"{chunk.core_start_sentence}-{chunk.core_end_sentence}")],
        ]
        metadata_table = Table(
            metadata_rows,
            colWidths=[30 * mm, 42 * mm, 30 * mm, 57 * mm],
        )
        metadata_table.setStyle(
            TableStyle(
                [
                    ("FONTNAME", (0, 0), (-1, -1), PDF_FONT),
                    ("FONTSIZE", (0, 0), (-1, -1), 8),
                    ("TEXTCOLOR", (0, 0), (-1, -1), colors.black),
                    ("GRID", (0, 0), (-1, -1), 0.35, colors.black),
                    ("BACKGROUND", (0, 0), (0, -1), colors.HexColor("#F2F2F2")),
                    ("BACKGROUND", (2, 0), (2, -1), colors.HexColor("#F2F2F2")),
                    ("VALIGN", (0, 0), (-1, -1), "TOP"),
                    ("LEFTPADDING", (0, 0), (-1, -1), 4),
                    ("RIGHTPADDING", (0, 0), (-1, -1), 4),
                    ("TOPPADDING", (0, 0), (-1, -1), 3),
                    ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
                ]
            )
        )
        story.extend(
            [
                metadata_table,
                Spacer(1, 7),
                Paragraph(escape(chunk.text), body_style),
            ]
        )

        if index < len(result.chunks) - 1:
            story.append(Spacer(1, 8))

    document.build(
        story,
        onFirstPage=_page_header_footer,
        onLaterPages=_page_header_footer,
    )


def save_chunking_json(result: ChunkingResult, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(
        json.dumps(result_to_dict(result), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

# 11. Integrity checks for every produced result

These checks ensure that the notebook does not silently produce broken chunk numbering, sentence coverage, segment metadata, or empty output files.

In [21]:
def validate_chunking_result(
    *,
    cleaned_text: str,
    result: ChunkingResult,
) -> None:
    assert result.chunks, "No chunks were produced."
    assert result.total_words == len(cleaned_text.split())
    assert [chunk.chunk_id for chunk in result.chunks] == list(
        range(1, len(result.chunks) + 1)
    )
    assert result.segment_count == len({chunk.segment_id for chunk in result.chunks})

    expected_core_start = 0
    for chunk in result.chunks:
        assert chunk.text.strip()
        assert chunk.word_count == len(chunk.text.split())
        assert chunk.core_start_sentence == expected_core_start
        assert chunk.core_end_sentence >= chunk.core_start_sentence
        expected_core_start = chunk.core_end_sentence + 1

        if chunk.is_continuation:
            assert chunk.continuation_of_chunk_id is not None
            assert chunk.continuation_reason == "max_size_split"
        else:
            assert chunk.continuation_of_chunk_id is None
            assert chunk.continuation_reason is None

    assert expected_core_start == result.total_sentences

In [23]:
# ================================================================
# MODULE 2 — SINGLE TRANSCRIPT CHUNKING TEST
#
# Input:
#   OUTPUT/Transcript/01_preprocessing.pdf
#
# Outputs:
#   OUTPUT/Transcript/02_chunking.pdf
#   OUTPUT/Transcript/02_chunking.json
#
# This cell does NOT rerun Module 1.
# ================================================================

from pathlib import Path
from collections import Counter
import json
import time


# ----------------------------------------------------------------
# 1. Configuration
# ----------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\Users\hp\EDTECH\Agent_1"
)

OUTPUT_DIR = PROJECT_ROOT / "OUTPUT"

TARGET_TRANSCRIPT_NAME = "Transcript"

transcript_folder = (
    OUTPUT_DIR
    / TARGET_TRANSCRIPT_NAME
)

module1_input = (
    transcript_folder
    / "01_preprocessing.pdf"
)

module2_pdf = (
    transcript_folder
    / "02_chunking.pdf"
)

module2_json = (
    transcript_folder
    / "02_chunking.json"
)


# ----------------------------------------------------------------
# 2. Check that Module 2 implementation cells were run
# ----------------------------------------------------------------

REQUIRED_MODULE2_OBJECTS = [
    "extract_final_cleaned_text_from_pdf",
    "final_chunker",
    "validate_chunking_result",
    "save_chunking_json",
    "export_chunking_pdf",
]

missing_objects = [
    object_name
    for object_name in REQUIRED_MODULE2_OBJECTS
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the previous Module 2 implementation cells first.\n\n"
        "Missing notebook objects:\n"
        + "\n".join(
            f"- {object_name}"
            for object_name in missing_objects
        )
    )


# ----------------------------------------------------------------
# 3. Validate input and output folders
# ----------------------------------------------------------------

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Agent_1 project folder was not found:\n"
        f"{PROJECT_ROOT}"
    )

if not OUTPUT_DIR.is_dir():
    raise FileNotFoundError(
        f"OUTPUT folder was not found:\n"
        f"{OUTPUT_DIR}"
    )

if not transcript_folder.is_dir():
    raise FileNotFoundError(
        f"Transcript output folder was not found:\n"
        f"{transcript_folder}\n\n"
        "Run the Module 1 single-transcript cell first."
    )

if not module1_input.is_file():
    raise FileNotFoundError(
        f"Module 1 PDF was not found:\n"
        f"{module1_input}\n\n"
        "Run Module 1 successfully before running Module 2."
    )

transcript_folder.mkdir(
    parents=True,
    exist_ok=True,
)


# ----------------------------------------------------------------
# 4. Record Module 1 file time
# ----------------------------------------------------------------

module1_original_mtime = (
    module1_input.stat().st_mtime_ns
)


# ----------------------------------------------------------------
# 5. Display paths
# ----------------------------------------------------------------

print("=" * 100)
print("MODULE 2 — SINGLE TRANSCRIPT TEST")
print("=" * 100)

print("Project root:")
print(PROJECT_ROOT)

print("\nTranscript folder:")
print(transcript_folder)

print("\nModule 1 input:")
print(module1_input)

print("\nModule 2 PDF output:")
print(module2_pdf)

print("\nModule 2 JSON output:")
print(module2_json)


# ----------------------------------------------------------------
# 6. Extract final cleaned text from Module 1 PDF
# ----------------------------------------------------------------

cleaned_text = (
    extract_final_cleaned_text_from_pdf(
        module1_input
    )
)

if not cleaned_text or not cleaned_text.strip():
    raise ValueError(
        "Module 1 PDF was found, but the final cleaned "
        "transcript section is empty."
    )

cleaned_text = cleaned_text.strip()

print("\nCleaned transcript characters:", len(cleaned_text))
print("Cleaned transcript words:", len(cleaned_text.split()))


# ----------------------------------------------------------------
# 7. Run existing MiniLM semantic chunking
# ----------------------------------------------------------------

started = time.perf_counter()

chunking_result = final_chunker.chunk(
    cleaned_text
)

runtime_seconds = (
    time.perf_counter()
    - started
)


# ----------------------------------------------------------------
# 8. Run existing integrity checks
# ----------------------------------------------------------------

validate_chunking_result(
    cleaned_text=cleaned_text,
    result=chunking_result,
)

if not chunking_result.chunks:
    raise RuntimeError(
        "Module 2 did not produce any chunks."
    )


# ----------------------------------------------------------------
# 9. Remove only old Module 2 outputs
# ----------------------------------------------------------------

for output_path in [
    module2_pdf,
    module2_json,
]:
    if output_path.exists():
        try:
            output_path.unlink()
        except PermissionError as error:
            raise PermissionError(
                f"The existing output is currently open:\n"
                f"{output_path}\n\n"
                "Close the file and run this cell again."
            ) from error


transcript_folder.mkdir(
    parents=True,
    exist_ok=True,
)


# ----------------------------------------------------------------
# 10. Save structured JSON for Module 3
# ----------------------------------------------------------------

save_chunking_json(
    chunking_result,
    module2_json,
)


# ----------------------------------------------------------------
# 11. Save readable Module 2 PDF
# ----------------------------------------------------------------

export_chunking_pdf(
    transcript_name=TARGET_TRANSCRIPT_NAME,
    source_pdf=module1_input,
    result=chunking_result,
    runtime_seconds=runtime_seconds,
    output_path=module2_pdf,
)


# ----------------------------------------------------------------
# 12. Verify both Module 2 outputs
# ----------------------------------------------------------------

for output_path in [
    module2_pdf,
    module2_json,
]:
    if not output_path.is_file():
        raise RuntimeError(
            f"Expected Module 2 output was not created:\n"
            f"{output_path}"
        )

    if output_path.stat().st_size == 0:
        raise RuntimeError(
            f"Module 2 output was created but is empty:\n"
            f"{output_path}"
        )


# ----------------------------------------------------------------
# 13. Validate saved JSON structure
# ----------------------------------------------------------------

saved_json_data = json.loads(
    module2_json.read_text(
        encoding="utf-8"
    )
)

saved_chunks = saved_json_data.get(
    "chunks",
    [],
)

if not isinstance(saved_chunks, list):
    raise TypeError(
        "02_chunking.json does not contain a valid chunks list."
    )

if len(saved_chunks) != len(chunking_result.chunks):
    raise AssertionError(
        "Saved JSON chunk count does not match the "
        "in-memory Module 2 result."
    )

for expected_chunk_id, saved_chunk in enumerate(
    saved_chunks,
    start=1,
):
    if saved_chunk.get("chunk_id") != expected_chunk_id:
        raise AssertionError(
            "Chunk IDs inside 02_chunking.json are not sequential."
        )

    if not str(saved_chunk.get("text", "")).strip():
        raise AssertionError(
            f"Chunk {expected_chunk_id} has empty text."
        )


# ----------------------------------------------------------------
# 14. Ensure Module 1 was not modified
# ----------------------------------------------------------------

if not module1_input.is_file():
    raise AssertionError(
        "Module 1 PDF was unexpectedly removed."
    )

if (
    module1_input.stat().st_mtime_ns
    != module1_original_mtime
):
    raise AssertionError(
        "Module 2 unexpectedly modified 01_preprocessing.pdf."
    )


# ----------------------------------------------------------------
# 15. Calculate readable metrics
# ----------------------------------------------------------------

chunk_word_counts = [
    chunk.word_count
    for chunk in chunking_result.chunks
]

boundary_reasons = Counter(
    chunk.boundary_reason
    for chunk in chunking_result.chunks
)

continuation_chunks = sum(
    bool(chunk.is_continuation)
    for chunk in chunking_result.chunks
)

overlap_chunks = sum(
    chunk.overlap_word_count > 0
    for chunk in chunking_result.chunks
)


# ----------------------------------------------------------------
# 16. Final summary
# ----------------------------------------------------------------

print("\n" + "=" * 100)
print("MODULE 2 COMPLETED SUCCESSFULLY")
print("=" * 100)

print("Runtime seconds:", round(runtime_seconds, 4))

print(
    "Embedding model:",
    chunking_result.embedding_model,
)

print(
    "Total words:",
    chunking_result.total_words,
)

print(
    "Total sentences:",
    chunking_result.total_sentences,
)

print(
    "Semantic units:",
    chunking_result.semantic_unit_count,
)

print(
    "Adaptive semantic threshold:",
    round(
        chunking_result.semantic_threshold,
        4,
    ),
)

print(
    "Physical chunks:",
    len(chunking_result.chunks),
)

print(
    "Logical segments:",
    chunking_result.segment_count,
)

print(
    "Continuation chunks:",
    continuation_chunks,
)

print(
    "Chunks containing overlap:",
    overlap_chunks,
)

print(
    "Minimum chunk words:",
    min(chunk_word_counts),
)

print(
    "Maximum chunk words:",
    max(chunk_word_counts),
)

print(
    "Average chunk words:",
    round(
        sum(chunk_word_counts)
        / len(chunk_word_counts),
        2,
    ),
)

print(
    "Boundary reasons:",
    dict(boundary_reasons),
)

print("\nGenerated files:")

print(
    f"- {module2_pdf} "
    f"({module2_pdf.stat().st_size:,} bytes)"
)

print(
    f"- {module2_json} "
    f"({module2_json.stat().st_size:,} bytes)"
)

print(
    "\nVerified: 01_preprocessing.pdf "
    "was not modified."
)


# ----------------------------------------------------------------
# 17. Show every generated chunk
# ----------------------------------------------------------------

print("\n" + "=" * 100)
print("GENERATED CHUNKS")
print("=" * 100)

for chunk in chunking_result.chunks:

    print("\n" + "-" * 100)

    print(
        f"Chunk {chunk.chunk_id} | "
        f"Segment {chunk.segment_id} | "
        f"Position {chunk.segment_position}"
    )

    print(
        f"Words: {chunk.word_count} | "
        f"Sentences: {chunk.sentence_count}"
    )

    print(
        f"Boundary: {chunk.boundary_reason} | "
        f"Similarity: {chunk.boundary_similarity}"
    )

    print(
        f"Continuation: {chunk.is_continuation} | "
        f"Continuation of: {chunk.continuation_of_chunk_id}"
    )

    print(
        f"Overlap words: {chunk.overlap_word_count}"
    )

    print("-" * 100)

    print(chunk.text)

MODULE 2 — SINGLE TRANSCRIPT TEST
Project root:
C:\Users\hp\EDTECH\Agent_1

Transcript folder:
C:\Users\hp\EDTECH\Agent_1\OUTPUT\Transcript

Module 1 input:
C:\Users\hp\EDTECH\Agent_1\OUTPUT\Transcript\01_preprocessing.pdf

Module 2 PDF output:
C:\Users\hp\EDTECH\Agent_1\OUTPUT\Transcript\02_chunking.pdf

Module 2 JSON output:
C:\Users\hp\EDTECH\Agent_1\OUTPUT\Transcript\02_chunking.json

Cleaned transcript characters: 686
Cleaned transcript words: 111

MODULE 2 COMPLETED SUCCESSFULLY
Runtime seconds: 0.0599
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Total words: 111
Total sentences: 7
Semantic units: 2
Adaptive semantic threshold: 0.2858
Physical chunks: 1
Logical segments: 1
Continuation chunks: 0
Chunks containing overlap: 0
Minimum chunk words: 111
Maximum chunk words: 111
Average chunk words: 111.0
Boundary reasons: {'end_of_transcript': 1}

Generated files:
- C:\Users\hp\EDTECH\Agent_1\OUTPUT\Transcript\02_chunking.pdf (43,881 bytes)
- C:\Users\hp\EDTECH\Agent_1\OUTP

# 12. Final batch: process all 12 Module 1 PDFs

This is the final production cell. It reads the 12 cleaned Module 1 PDFs from `OUTPUT`, applies the selected MiniLM chunking logic, and writes each result into the same transcript folder.

In [16]:
def process_all_cleaned_transcripts() -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []

    for index, input_pdf in enumerate(module1_pdf_paths, start=1):
        transcript_name = input_pdf.parent.name
        output_pdf = input_pdf.parent / MODULE2_OUTPUT_PDF
        output_json = input_pdf.parent / MODULE2_OUTPUT_JSON

        if (
            not OVERWRITE_EXISTING_OUTPUTS
            and output_pdf.exists()
            and output_json.exists()
        ):
            rows.append(
                {
                    "transcript": transcript_name,
                    "status": "skipped_existing",
                    "input_pdf": str(input_pdf),
                    "output_pdf": str(output_pdf),
                    "output_json": str(output_json),
                }
            )
            continue

        row: dict[str, Any] = {
            "transcript": transcript_name,
            "status": "running",
            "input_pdf": str(input_pdf),
            "output_pdf": str(output_pdf),
            "output_json": str(output_json),
        }

        try:
            cleaned_text = extract_final_cleaned_text_from_pdf(input_pdf)
            start = time.perf_counter()
            result = final_chunker.chunk(cleaned_text)
            runtime = time.perf_counter() - start

            validate_chunking_result(cleaned_text=cleaned_text, result=result)
            save_chunking_json(result, output_json)
            export_chunking_pdf(
                transcript_name=transcript_name,
                source_pdf=input_pdf,
                result=result,
                runtime_seconds=runtime,
                output_path=output_pdf,
            )

            if output_pdf.stat().st_size == 0 or output_json.stat().st_size == 0:
                raise RuntimeError("A Module 2 output file was created empty.")

            row.update(
                {
                    "status": "completed",
                    "runtime_seconds": round(runtime, 4),
                    **result_metrics(result),
                }
            )

            print(
                f"[{index:02d}/{len(module1_pdf_paths)}] COMPLETED | "
                f"{transcript_name:<50} | "
                f"{len(result.chunks):>3} chunks | "
                f"{result.segment_count:>3} segments | "
                f"{runtime:.3f}s"
            )

        except Exception as exc:
            row.update(
                {
                    "status": "failed",
                    "error_type": type(exc).__name__,
                    "error_message": str(exc),
                }
            )
            print(
                f"[{index:02d}/{len(module1_pdf_paths)}] FAILED    | "
                f"{transcript_name:<50} | {type(exc).__name__}: {exc}"
            )

        rows.append(row)

    return rows


batch_results = process_all_cleaned_transcripts()

[01/1] COMPLETED | Transcript                                         |   1 chunks |   1 segments | 0.155s


# 13. Save batch summary and verify all 12 outputs

In [17]:
MODULE2_BATCH_JSON = OUTPUT_DIR / "module2_batch_summary.json"
MODULE2_BATCH_CSV = OUTPUT_DIR / "module2_batch_summary.csv"

MODULE2_BATCH_JSON.write_text(
    json.dumps(batch_results, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

try:
    import pandas as pd

    batch_dataframe = pd.DataFrame(batch_results)
    batch_dataframe.to_csv(MODULE2_BATCH_CSV, index=False)
    display(batch_dataframe)
except Exception:
    batch_dataframe = None
    import csv

    keys = sorted({key for row in batch_results for key in row})
    with MODULE2_BATCH_CSV.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=keys)
        writer.writeheader()
        writer.writerows(batch_results)

completed_rows = [row for row in batch_results if row["status"] == "completed"]
failed_rows = [row for row in batch_results if row["status"] == "failed"]

assert not failed_rows, json.dumps(failed_rows, indent=2, ensure_ascii=False)
assert len(completed_rows) == EXPECTED_TRANSCRIPT_COUNT

for input_pdf in module1_pdf_paths:
    output_pdf = input_pdf.parent / MODULE2_OUTPUT_PDF
    output_json = input_pdf.parent / MODULE2_OUTPUT_JSON
    assert output_pdf.exists() and output_pdf.stat().st_size > 0
    assert output_json.exists() and output_json.stat().st_size > 0

    # Verify that the generated PDF can be opened and has at least one page.
    with fitz.open(output_pdf) as document:
        assert document.page_count >= 1

print("\n" + "=" * 100)
print("MODULE 2 BATCH COMPLETED SUCCESSFULLY")
print("=" * 100)
print(f"Completed transcripts: {len(completed_rows)}")
print(f"Failed transcripts: {len(failed_rows)}")
print(f"Summary JSON: {MODULE2_BATCH_JSON}")
print(f"Summary CSV: {MODULE2_BATCH_CSV}")
print("Each transcript folder now contains:")
print(f"- {MODULE1_INPUT_PDF}")
print(f"- {MODULE2_OUTPUT_PDF}")
print(f"- {MODULE2_OUTPUT_JSON}")

,transcript,status,input_pdf,output_pdf,output_json,runtime_seconds,embedding_model,total_words,total_sentences,semantic_units,semantic_threshold,physical_chunks,logical_segments,continuation_chunks,overlap_chunks,minimum_chunk_words,maximum_chunk_words,average_chunk_words,boundary_reasons
0,Transcript,completed,C:\Users\hp\EDTECH\Agent_1\OUTPUT\Transcript\0...,C:\Users\hp\EDTECH\Agent_1\OUTPUT\Transcript\0...,C:\Users\hp\EDTECH\Agent_1\OUTPUT\Transcript\0...,0.1549,sentence-transformers/all-MiniLM-L6-v2,100,5,2,0.45,1,1,0,0,100,100,100.0,{'end_of_transcript': 1}


AssertionError: 

# 14. Final migration note

After this notebook runs successfully, Module 2 no longer depends on its old Python implementation files. Keep `02_chunking.json` because Module 3 can consume the structured chunks directly; `02_chunking.pdf` is the readable review output.